# 02 - Reading SPINE Output

Goal: open a reconstructed SPINE HDF5 file, inspect the high-level object hierarchy, and connect particle and interaction fields to analysis questions.

This notebook is the workshop on-ramp. It moves step by step through the basic SPINE object model and the main event-level collections.

## Before you type anything

Start with this mental model:

- A SPINE output file is a collection of events.
- Each event contains a mixture of reconstructed objects and/or truth objects.
- The most common high-level objects are particles and interactions.
- A particle is one logical reconstructed physics object:
  - In the case of single track-like particles, it is all depositions associated with the same track.
  - In the case of shower-like particles, it is a collection of shower fragments that originate from the same primary particle.
- An interaction is a group of particles that belong together (originate from the same primary vertex).

You do not need to memorize the full object model. The goal is to learn how to inspect it easily.

## Runtime contract

Run inside `ghcr.io/deeplearnphysics/spine:latest` (see `00_eaf_setup.md`).

Set the sample in the first code cell:

```python
DETECTOR = "2x2"
SAMPLE_NAME = "2x2_numi"
GEOMETRY = DETECTOR
```

The notebooks look for data in this order:

1. `/exp/dune/data/users/drielsma/npc-ddas`
2. `tutorials/assets`

Internally the code checks both `../assets` and `tutorials/assets` so the same repo-local fallback still works when the notebook kernel starts in a different working directory.

In either case they read:

```python
HDF5_DATA_DIR / DETECTOR / f"{SAMPLE_NAME}_spine.h5"
```

That keeps the workshop default aligned with the shared DUNE location while still allowing a repo-local fallback.

The HDF5 file should contain reconstructed particles and interactions and, for later validation, truth objects.

## Step 1: import the tools for this notebook

This first code cell does not touch any data yet. It only imports the Python modules we will use later.

If an import fails, stop here. There is no point debugging later cells until this environment cell runs cleanly.

In [ ]:

# Standard Python/path tools for working with local files.
from pathlib import Path

# Analysis utilities used throughout the tutorials.
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml

# The Driver is the main SPINE entry point for reading one event at a time.
from spine.driver import Driver


## Step 2: choose the input file

This second code cell answers one question: which SPINE HDF5 file should the rest of the notebook read?

It looks for the shared workshop area first, then for the repository-local tutorial assets. It also defines the detector name and sample tag that control which file gets opened.

In [ ]:

# Prefer the shared DUNE path used in the workshop environment.
# Fall back to the repository-local tutorial assets when needed.
# We include both relative spellings because notebook kernels do not always
# start in the notebook directory.
DATA_ROOT_CANDIDATES = [
    Path("/exp/dune/data/users/drielsma/npc-ddas"),
    Path("../assets"),
    Path("tutorials/assets"),
]

# Pick the first location that actually exists on disk.
DATA_ROOT = next((path for path in DATA_ROOT_CANDIDATES if path.exists()), None)
if DATA_ROOT is None:
    raise RuntimeError("Could not find a workshop data directory.")

# The reco HDF5 files live under reco/ inside the chosen data root.
HDF5_DATA_DIR = DATA_ROOT / "reco"

# Edit these values to switch samples.
# The expected layout is:
#   reco/DETECTOR/SAMPLE_NAME_spine.h5
DETECTOR = "2x2"
SAMPLE_NAME = "2x2_numi"

# GEOMETRY tells SPINE which detector geometry description to use.
# For this workshop we use the detector name directly.
GEOMETRY = DETECTOR
HDF5_FILE_NAME = f"{SAMPLE_NAME}_spine.h5"

# Build the final path to the reconstructed SPINE HDF5 file.
DATA_PATH = HDF5_DATA_DIR / DETECTOR / HDF5_FILE_NAME

print(f"Using data root: {DATA_ROOT}")
print(f"Reco file: {DATA_PATH}")


## Step 3: build the SPINE driver from YAML

Now that the notebook knows which file to read, it loads the tutorial YAML config, injects the concrete file path, and creates a `Driver` object.

The geometry override in this step is important: it tells SPINE which detector geometry description to attach when the chosen sample needs one.

In [ ]:

# As with the data paths, the notebook working directory can vary.
# These two candidates point to the same tutorial config from different cwd values.
CONFIG_CANDIDATES = [
    Path("../config/read_spine_hdf5.yaml"),
    Path("tutorials/config/read_spine_hdf5.yaml"),
]
CONFIG_PATH = next((path for path in CONFIG_CANDIDATES if path.exists()), None)

# Replace the DATA_PATH placeholder in the YAML template with the file we chose above.
DATA_PATH = str(DATA_PATH)
if CONFIG_PATH is None:
    raise RuntimeError("Could not find tutorials/config/read_spine_hdf5.yaml")

# Read the YAML as text first so we can substitute the concrete file path.
cfg_text = CONFIG_PATH.read_text().replace("DATA_PATH", DATA_PATH)

# Convert the YAML text into a normal Python dictionary.
cfg = yaml.safe_load(cfg_text)

# Some detector samples need an explicit geometry block so downstream code knows
# which detector layout and coordinate conventions to use.
if GEOMETRY:
    cfg["geo"] = {"detector": GEOMETRY}

# Create the SPINE driver. From this point on, driver.process(entry=...) reads events.
driver = Driver(cfg)
print(f"Opened {DATA_PATH}")
print(f"Entries: {len(driver)}")
if GEOMETRY:
    print(f"Detector geometry: {GEOMETRY}")


## Config loading note: `safe_load` vs `load_config_file`

The setup cell above uses `yaml.safe_load` because this tutorial config is deliberately tiny: it reads one YAML template, replaces `DATA_PATH`, and creates a normal Python dictionary.

For real SPINE configuration work, prefer `spine.config.load_config_file`. That loader understands SPINE's config composition features, including `include`, `!include`, `!path`, `!download`, `override`, and removal operations. It is the better tool when you want to start from a production config and add or modify blocks in a notebook.

For example, you can load a base config and add a block from Python:

```python
cfg = load_config_file("/path/to/base_config.yaml")
cfg["geo"] = {"detector": DETECTOR}
cfg["ana"] = {"save": save_cfg}
driver = Driver(cfg)
```

If you prefer to express the same geometry change in YAML instead of Python, you can make that edit directly in the config layer:

```yaml
include: /path/to/base_config.yaml

geo:
    detector: DETECTOR
```

That YAML overlay is equivalent in spirit to loading the base config and then setting `cfg["geo"]` from Python.

Use `help(load_config_file)` when you want to explore its options interactively.

## The config we are actually using

The driver is not magic. It is created from a YAML config. For this notebook we inspect it once, here, and later notebooks will refer back to this pattern instead of repeating the full dump.

Read this cell carefully. It tells you what the notebook is loading and which matching post-processors are enabled.

In [ ]:
print(cfg_text)

# If you prefer Python objects to raw YAML, inspect `cfg` as well.
cfg

## Where to look things up

When a field or class is unfamiliar, use three complementary tools:

- Python introspection: `help(obj)`, `dir(obj)`, `obj.as_dict().keys()`
- The SPINE API browser: https://spine.readthedocs.io
- Source and config examples in `spine-prod`

The point is not to memorize every field. The point is to learn how to inspect the object model without guessing.

## Read one entry

Start with exactly one event entry. Before running the next cell, make a prediction:

1. What do you think `driver.process(...)` returns?
2. Do you expect a list, a dictionary, or one custom object?
3. What keys do you think will be present?

In [ ]:
ENTRY = 0

# `driver.process` returns a dictionary of event-level data products.
data = driver.process(entry=ENTRY)

In [ ]:
list(data)

Now pull out the four high-level collections we will use most often. Pause here and check the counts before inspecting individual objects.

If one collection is empty, that is not automatically a bug. It may just mean the file does not contain that category for this event.

In [ ]:
reco_particles = data.get("reco_particles", [])
truth_particles = data.get("truth_particles", [])
reco_interactions = data.get("reco_interactions", [])
truth_interactions = data.get("truth_interactions", [])

In [ ]:
pd.DataFrame(
    {
        "collection": [
            "reco_particles",
            "truth_particles",
            "reco_interactions",
            "truth_interactions",
        ],
        "count": [
            len(reco_particles),
            len(truth_particles),
            len(reco_interactions),
            len(truth_interactions),
        ],
    }
)

## Inspect one object slowly

SPINE objects expose Python attributes and usually an `as_dict()` method. Start by looking at one reconstructed particle and one reconstructed interaction.

This is the moment where beginners usually start guessing field names. Do not guess. Inspect.

In [ ]:
particle = reco_particles[0]
interaction = reco_interactions[0]

In [ ]:
type(particle), type(interaction)

In [ ]:
# `as_dict()` gives a fast survey of what the object knows about itself.
list(particle.as_dict())

In [ ]:
# Try one or more of these during the tutorial.
# help(particle)
# dir(particle)
# particle.as_dict().keys()
# interaction.as_dict().keys()

In [ ]:
def preview_value(value):
    if isinstance(value, np.ndarray):
        return f"array shape={value.shape} dtype={value.dtype}"
    return repr(value)[:120]


def compact_dict(obj, max_items=25):
    if hasattr(obj, "as_dict"):
        mapping = obj.as_dict()
    else:
        mapping = vars(obj)
    rows = [(key, preview_value(value)) for key, value in list(mapping.items())[:max_items]]
    return pd.DataFrame(rows, columns=["field", "preview"])


compact_dict(particle)

In [ ]:
compact_dict(interaction)

## Micro-exercise: ask the object questions

Use the previous output to answer these before moving on:

1. Which field appears to store the particle ID?
2. Which field appears to link a particle to an interaction?
3. Which field looks like a particle type prediction?
4. Which field looks energy-like or charge-like?

If you are not sure, test your guess with one tiny line of code.

In [ ]:
{
    "particle_id": getattr(particle, "id", None),
    "interaction_id": getattr(particle, "interaction_id", None),
    "pid": getattr(particle, "pid", None),
    "depositions_sum": getattr(particle, "depositions_sum", None),
}

## Use SPINE labels

Do not hard-code PID or semantic-shape labels in analysis notebooks. Import the labels from SPINE so the notebook follows the installed package.

In [ ]:
from spine.constants import PID_LABELS, SHAPE_LABELS

In [ ]:
display(pd.Series(SHAPE_LABELS, name="shape label").to_frame())
display(pd.Series(PID_LABELS, name="PID label").to_frame())

## Build a first particle table

Pick a short list of fields first. This is the analysis contract you are choosing to rely on.

Notice how small this list is. In real analysis you almost never want every field at once.

In [ ]:
PARTICLE_FIELDS = [
    "id",
    "interaction_id",
    "pid",
    "shape",
    "is_primary",
    "size",
    "depositions_sum",
]

PARTICLE_FIELDS

In [ ]:
# Build one row per reconstructed particle.
particle_df = pd.DataFrame(
    [{field: getattr(p, field, None) for field in PARTICLE_FIELDS} for p in reco_particles]
)
particle_df["pid_label"] = particle_df["pid"].map(PID_LABELS)
particle_df["shape_label"] = particle_df["shape"].map(SHAPE_LABELS)
particle_df

## Micro-exercise: answer real analysis questions

Try to answer each question with one or two lines of Python:

1. What is the PID label of the first reconstructed particle?
2. Which particles belong to the same interaction as that particle?
3. Which of those particles are marked primary?
4. Which column would you use as a first-pass energy or charge proxy?

In [ ]:
first_interaction_id = particle_df.loc[0, "interaction_id"]
first_interaction_id

In [ ]:
same_interaction_particles = particle_df.query("interaction_id == @first_interaction_id")
same_interaction_particles

## Build an interaction table

Interactions group particles. The next table summarizes multiplicities and primary content without forcing us to inspect every particle manually.

In [ ]:
def primary_particles(interaction):
    return [p for p in getattr(interaction, "particles", []) if getattr(p, "is_primary", False)]

In [ ]:
interaction_df = pd.DataFrame(
    [
        {
            "id": getattr(ia, "id", None),
            "nu_id": getattr(ia, "nu_id", None),
            "size": getattr(ia, "size", None),
            "vertex": getattr(ia, "vertex", None),
            "n_particles": len(getattr(ia, "particles", [])),
            "n_primary": len(primary_particles(ia)),
            "primary_pids": [getattr(p, "pid", None) for p in primary_particles(ia)],
            "topology": getattr(ia, "topology", None),
        }
        for ia in reco_interactions
    ]
)
interaction_df

## Cross-check particle to interaction bookkeeping

The next cell answers a common beginner question: if a particle stores an `interaction_id`, can we recover the full interaction and compare the counts?

In [ ]:
chosen_interaction_id = int(first_interaction_id)

interaction_df.query("id == @chosen_interaction_id")

## Live exercise

Pick one interaction and answer:

- Which primary particles does SPINE reconstruct?
- Is the vertex close to the visually obvious interaction point?
- If you wanted the total deposited charge for this interaction, would you sum particle-level values or read a field from the interaction object? Why?

In [ ]:
from spine.vis.out import Drawer

drawer = Drawer(data)
fig = drawer.get("interactions", "id")
fig.show()

## Inference belongs in `spine-prod`

This short tutorial starts from HDF5 output. For real production, `spine-prod` should own the validated full-chain inference configs, model weights, campaign metadata, and file bookkeeping. That lets analysis notebooks stay focused on object interpretation and validation.

## Offline extensions

- Repeat the field inspection for a second detector sample and list which object fields are detector-dependent.
- Build a small field glossary for the 10 particle and interaction attributes your analysis will use.
- Compare the same event in this notebook and Spinal Tap, then record which table fields explain the visual topology.
- Write one extra cell that computes the total `depositions_sum` for every interaction by grouping particles.